In [ ]:

import os
import hashlib
from collections import Counter
from typing import List, Union
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import tqdm
from sklearn.metrics import classification_report
from google.colab import drive
import subprocess

# Install timm library if not present
try:
    import timm
except ImportError:
    subprocess.run(["pip", "install", "-q", "timm"])
    import timm

# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except ImportError:
    print("Not running in Google Colab. Skipping Drive mount.")

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


In [ ]:

# --- Paths (Aligned with EfficientNet & SE-ResNeXt) ---
subprocess.run(["unzip", 
    "-q", "/content/drive/MyDrive/Datasets/kaggle_knee_osteoarthritis.zip",
    "-d", "/content/Datasets"])

DATASET_ROOT_PATH = "/content/Datasets/kaggle_knee_osteoarthritis"
CHECKPOINT_SAVE_DIR = "/content/drive/MyDrive/Models/densenet121_checkpoints"

os.makedirs(CHECKPOINT_SAVE_DIR, exist_ok=True)

# --- Training Hyperparameters ---
EPOCHS = 20
BATCH_SIZE = 16
IMG_SIZE = 224  # Standard size for DenseNet-121
INITIAL_LR = 1e-4
WEIGHT_DECAY = 1e-4


In [ ]:

class SquarePadOpenCV(object):
    """Pads a rectangular image to a square."""
    def __call__(self, image):
        h, w = image.shape[:2]
        max_wh = max(h, w)
        pad_top = (max_wh - h) // 2
        pad_bottom = max_wh - h - pad_top
        pad_left = (max_wh - w) // 2
        pad_right = max_wh - w - pad_left
        
        padded_image = cv2.copyMakeBorder(
            image, pad_top, pad_bottom, pad_left, pad_right, 
            borderType=cv2.BORDER_CONSTANT, value=[0, 0, 0]
        )
        return padded_image

class OpenCVCLAHE(object):
    """Applies CLAHE (Contrast Limited Adaptive Histogram Equalization) using OpenCV."""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img_rgb: np.ndarray) -> np.ndarray:
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        l_channel, a_channel, b_channel = cv2.split(img_lab)
        clahe_l_channel = clahe.apply(l_channel)
        merged_lab_image = cv2.merge((clahe_l_channel, a_channel, b_channel))
        return cv2.cvtColor(merged_lab_image, cv2.COLOR_LAB2RGB)

def get_transforms(img_size=224):
    """Returns training and validation transforms."""
    train_transform = transforms.Compose([
        SquarePadOpenCV(),
        OpenCVCLAHE(),
        transforms.ToPILImage(),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.80, 1.20), shear=5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        SquarePadOpenCV(),
        OpenCVCLAHE(),
        transforms.ToPILImage(),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_transform, val_transform

def remove_duplicate_images(image_paths: List[str], labels: List[int], exclude_hashes: set = None, categories: List[str] = None):
    """Removes duplicate images using MD5 hashing."""
    total_found = len(image_paths)
    unique_paths, unique_labels, unique_hashes = [], [], set()
    internal_dup_count, leakage_count = 0, 0
    
    for path, label in zip(image_paths, labels):
        hash_md5 = hashlib.md5()
        try:
            with open(path, "rb") as f:
                for chunk in iter(lambda: f.read(4096), b""):
                    hash_md5.update(chunk)
            h = hash_md5.hexdigest()
        except Exception as e:
            print(f"Warning: Could not read image {path}: {e}")
            continue
            
        if exclude_hashes and h in exclude_hashes:
            leakage_count += 1
            continue
        if h in unique_hashes:
            internal_dup_count += 1
            continue
            
        unique_hashes.add(h)
        unique_paths.append(path)
        unique_labels.append(label)
        
    class_counts = Counter(unique_labels)
    print(f"\n--- Dataset Statistics & Deduplication ---")
    print(f"  - Total files: {total_found} | Unique kept: {len(unique_paths)}")
    print(f"  - Internal dupes removed: {internal_dup_count} | Cross-split leaks removed: {leakage_count}")
    return unique_paths, unique_labels, unique_hashes

class KaggleKneeOsteoarthritisDataset(Dataset):
    """Dataset class specifically for the Kaggle Knee Osteoarthritis dataset."""
    def __init__(self, root: str, split_dir: str, transform=None, exclude_hashes: set = None):
        self.root = root
        self.transform = transform
        self.exclude_hashes = exclude_hashes
        raw_paths, raw_labels = [], []
        split_path = os.path.join(root, split_dir)
        
        if not os.path.isdir(split_path): 
            raise FileNotFoundError(f"Split directory not found: {split_path}")
            
        class_names = sorted([d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d)) and d.isdigit()])
        print(f"Loading '{split_dir}' split from: {split_path}")
        
        for class_name in class_names:
            class_dir = os.path.join(split_path, class_name)
            label = int(class_name)
            valid_extensions = ('.png', '.jpg', '.jpeg')
            image_files = [f for f in os.listdir(class_dir) if f.lower().endswith(valid_extensions)]
            for file_name in image_files:
                raw_paths.append(os.path.join(class_dir, file_name))
                raw_labels.append(label)
                
        self.image_paths, self.labels, self.image_hashes = remove_duplicate_images(
            raw_paths, raw_labels, exclude_hashes=self.exclude_hashes, categories=class_names
        )

    def load_image_from_path(self, image_path: str) -> np.ndarray:
        img_bgr = cv2.imread(image_path)
        if img_bgr is None: raise IOError(f"Could not read image: {image_path}")
        return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx: int):
        image = self.load_image_from_path(self.image_paths[idx])
        label = self.labels[idx]
        if self.transform: image = self.transform(image)
        return image, label

    def __len__(self) -> int: 
        return len(self.image_paths)


In [ ]:

# --- 1. Prepare Data Loaders ---
train_transform, val_transform = get_transforms(img_size=IMG_SIZE)

train_dataset = KaggleKneeOsteoarthritisDataset(root=DATASET_ROOT_PATH, split_dir="train", transform=train_transform)
train_hashes = set(train_dataset.image_hashes)

val_split_dir = "val" if os.path.isdir(os.path.join(DATASET_ROOT_PATH, "val")) else "test"
val_dataset = KaggleKneeOsteoarthritisDataset(root=DATASET_ROOT_PATH, split_dir=val_split_dir, transform=val_transform, exclude_hashes=train_hashes)

# Dynamic split if val dataset is empty after leakage removal
if len(val_dataset) == 0:
    import random
    print("Performing dynamic 80/20 train/validation split...")
    combined = list(zip(train_dataset.image_paths, train_dataset.labels))
    random.seed(42)
    random.shuffle(combined)
    split_idx = int(len(combined) * 0.8)
    train_pairs, val_pairs = combined[:split_idx], combined[split_idx:]
    
    train_dataset.image_paths, train_dataset.labels = [p for p, _ in train_pairs], [l for _, l in train_pairs]
    val_dataset.image_paths, val_dataset.labels = [p for p, _ in val_pairs], [l for _, l in val_pairs]
    print(f"Post-Split - Train: {len(train_dataset)}, Val: {len(val_dataset)}")

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:

class CoralLayer(nn.Module):
    """Custom CORAL weight-sharing layer for consistent rank logits."""
    def __init__(self, size_in, num_classes):
        super(CoralLayer, self).__init__()
        self.size_in = size_in
        self.num_classes = num_classes
        self.coral_weights = nn.Parameter(torch.Tensor(size_in, 1))
        self.coral_bias = nn.Parameter(torch.Tensor(num_classes - 1))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.coral_weights)
        nn.init.zeros_(self.coral_bias)

    def forward(self, x):
        logits = torch.matmul(x, self.coral_weights) + self.coral_bias
        return logits

class DenseNetModel(nn.Module):
    def __init__(self, num_classes: int = 5, pretrained: bool = True):
        super(DenseNetModel, self).__init__()
        self.num_classes = num_classes
        # Load DenseNet-121 backbone without output classification head
        self.backbone = timm.create_model('densenet121', pretrained=pretrained, num_classes=0)
        # DenseNet-121 features output size is 1024
        self.coral_layer = CoralLayer(size_in=1024, num_classes=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        logits = self.coral_layer(features)
        return logits

    def fit(self, epoch, data_loader, optimizer, device, criterion=None):
        self.to(device)
        self.train()
        running_loss, total, correct = 0.0, 0, 0
        
        progress_bar = tqdm.tqdm(data_loader, desc=f"Epoch {epoch+1} [TRAIN]")
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = self(images)

            # Compute CORAL Loss (Average BCE across 4 sub-tasks)
            loss = 0.0
            num_tasks = self.num_classes - 1
            for j in range(num_tasks):
                targets_j = (labels > j).float()
                loss += F.binary_cross_entropy_with_logits(output[:, j], targets_j)
            loss = loss / num_tasks

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            
            # Predict class: sum of task probabilities exceeded threshold 0.5
            probs = torch.sigmoid(output)
            predicted = (probs > 0.5).sum(dim=1)
            
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{100.0 * correct / total:.2f}%"
            })
            
        return running_loss / total, 100.0 * correct / total

    def evaluate(self, epoch, data_loader, device, criterion=None):
        self.to(device)
        self.eval()
        running_loss, total, correct = 0.0, 0, 0
        all_preds, all_labels = [], []
        
        progress_bar = tqdm.tqdm(data_loader, desc=f"Epoch {epoch+1} [VALIDATE]")
        with torch.no_grad():
            for images, labels in progress_bar:
                images, labels = images.to(device), labels.to(device)
                output = self(images)

                # Compute CORAL Loss (Average BCE)
                loss = 0.0
                num_tasks = self.num_classes - 1
                for j in range(num_tasks):
                    targets_j = (labels > j).float()
                    loss += F.binary_cross_entropy_with_logits(output[:, j], targets_j)
                loss = loss / num_tasks

                running_loss += loss.item() * images.size(0)
                
                # Predict class
                probs = torch.sigmoid(output)
                predicted = (probs > 0.5).sum(dim=1)
                
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        report = classification_report(
            all_labels, all_preds, 
            target_names=[str(i) for i in range(5)], 
            zero_division=0
        )
        return running_loss / total, 100.0 * correct / total, report


In [ ]:

# --- 2. Initialize Model (Aligned config) ---
model = DenseNetModel(num_classes=5, pretrained=True)

# AdamW optimizer with weight decay
optimizer = optim.AdamW(model.parameters(), lr=INITIAL_LR, weight_decay=WEIGHT_DECAY)

last_model_path = os.path.join(CHECKPOINT_SAVE_DIR, "last_model.pth")
best_model_path = os.path.join(CHECKPOINT_SAVE_DIR, "best_model.pth")


In [ ]:

class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pt'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model, epoch):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model, epoch)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model, epoch)
            self.counter = 0
        return self.early_stop

    def save_checkpoint(self, val_loss, model, epoch):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        checkpoint = {
            "model": model.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss
        }
        torch.save(checkpoint, self.path)
        self.val_loss_min = val_loss


In [ ]:

early_stopper = EarlyStopping(patience=10, verbose=True, path=best_model_path)

for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    train_loss, train_acc = model.fit(epoch, train_loader, optimizer, device)
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")

    val_loss, val_acc, report = model.evaluate(epoch, val_loader, device)
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    print(report)
    
    # Save last epoch checkpoint
    checkpoint = {
        "model": model.state_dict(),
        "epoch": epoch,
        "val_loss": val_loss
    }
    torch.save(checkpoint, last_model_path)
    
    if early_stopper(val_loss, model, epoch):
        print("Early stopping triggered!")
        break


In [ ]:

def predict_single_image_opencv(image_path, model, device, transform):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None: raise IOError(f"Could not read image: {image_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_tensor = transform(img_rgb)
    img_batch = img_tensor.unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        logits = model(img_batch)
        probs_gt = torch.sigmoid(logits).cpu().numpy()[0] # Shape [4]
        
    predicted_idx = int(np.sum(probs_gt > 0.5))
    
    # Convert cumulative probabilities to individual class probabilities
    p = np.zeros(5)
    p[0] = 1.0 - probs_gt[0]
    p[1] = probs_gt[0] - probs_gt[1]
    p[2] = probs_gt[1] - probs_gt[2]
    p[3] = probs_gt[2] - probs_gt[3]
    p[4] = probs_gt[3]
    p = np.clip(p, 0.0, 1.0)
    p_sum = np.sum(p)
    if p_sum > 0:
        p = p / p_sum
    else:
        p = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
        
    categories = ['0Normal', '1Doubtful', '2Mild', '3Moderate', '4Severe']
    return categories[predicted_idx], p[predicted_idx] * 100.0


In [ ]:

def load_model_from_weight_file(model, path, device):
    checkpoint = torch.load(path, map_location=device)
    if isinstance(checkpoint, dict) and "model" in checkpoint:
        model.load_state_dict(checkpoint["model"])
    else:
        model.load_state_dict(checkpoint)
    model.eval()
    return model.to(device)


In [ ]:

model = DenseNetModel(num_classes=5, pretrained=False)
if os.path.exists(best_model_path):
    model = load_model_from_weight_file(model, best_model_path, device)
    print("Loaded model weights successfully.")
else:
    print(f"Warning: Weight file not found at {best_model_path}.")


In [ ]:

# Test classification prediction on a sample image if available
image_test = "/content/Datasets/kaggle_knee_osteoarthritis/val/4/9029707R.png"
if os.path.exists(image_test):
    val_transform = get_transforms(img_size=IMG_SIZE)[1]
    predicted_class, confidence_score = predict_single_image_opencv(image_test, model, device, val_transform)
    print(f"Predicted Class: {predicted_class}")
    print(f"Confidence Score: {confidence_score:.2f}%")
else:
    print("Test image not found, skip inference test.")


In [ ]:

# Disconnect Colab runtime to save credits after training finishes
try:
    from google.colab import runtime
    print("Training complete. Disconnecting runtime...")
    runtime.unassign()
except ImportError:
    print("Not running in Colab. Skip unassign.")
